In [ ]:
import psycopg2
import pandas as pd

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

In [ ]:
df = df_original.copy(deep=True)
df = df[
    (df['dept_name'] == 'Power-System') & (df['status_id'] == 1)
][['filename', 'workorder_id', 'interval', 'dept_name', 'json_data']]

df

In [ ]:
import json

def normalize(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return None
    return None

df["json_data"] = df["json_data"].apply(normalize)

In [ ]:
def safe_get(d, keys):
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d

In [ ]:
def find_key(data, target_key):
    if isinstance(data, dict):
        for k, v in data.items():
            if k == target_key:
                return v
            result = find_key(v, target_key)
            if result is not None:
                return result

    elif isinstance(data, list):
        for item in data:
            result = find_key(item, target_key)
            if result is not None:
                return result

    return None

extracted_df = pd.DataFrame({

    "workorder_no": df["workorder_id"],
    
    "inspection_date": df["json_data"].apply(lambda x: find_key(x, "inspection_date")),
    "inspection_time": df["json_data"].apply(lambda x: find_key(x, "inspection_time")),
    
    "perform_by": df["json_data"].apply(lambda x: find_key(x, "perform_by")),

    "verify_by": df["json_data"].apply(lambda x: find_key(x, "verify_by")),
    
    "filename": df["filename"],

})

extracted_df.head()

In [ ]:
def extract_id_and_date(value):
    if isinstance(value, dict):
        return value.get("id"), value.get("date")
    return None, None
extracted_df[["technician_ids", "technician_date"]] = extracted_df["perform_by"].apply(lambda x: pd.Series(extract_id_and_date(x)))
extracted_df[["supervisor_id", "supervisor_date"]] = extracted_df["verify_by"].apply(lambda x: pd.Series(extract_id_and_date(x)))

extracted_df.head()

In [ ]:
extracted_df.to_excel("extracted/extracted_psd.xlsx", index=False)

### Comparing list of technician and supervisor with tbl_user

In [ ]:
ref_user = pd.read_excel("tbl_users.xlsx")
ref_user.head()

In [ ]:
cleaned_user = ref_user[["staff_id", "name", "call_sign", "department"]].copy()

cleaned_user["stamp_id"] = (
    cleaned_user["staff_id"]
    .astype(str)
    .str.replace(r"^1000|^100", "", regex=True)
)

cleaned_user["stamp_id"] = cleaned_user["stamp_id"].astype(str)
cleaned_user.head()

In [1]:
import pandas as pd

df_psd = pd.read_excel("extracted/extracted_psd.xlsx")
df_psd.head()

,workorder_no,inspection_date,inspection_time,perform_by,verify_by,filename,technician_ids,technician_date,supervisor_id,supervisor_date
0,4000495459,2022-11-01 00:00:00,"{'start_time': '1000', 'end_time': '1100'}","{'name': 'NAZRUL SHAH', 'id': 11858, 'technici...","{'name': 'ABDUL TOLIB', 'id': 7099, 'technicia...",PS_PM_WEK_StationInspection_4000495459.pdf,11858,2022-11-01 00:00:00,7099,01/11/2022
1,4000530204,2023-05-09 00:00:00,"{'start_time': '0900', 'end_time': 1000}","{'name': 'SHAZWAN', 'id': '12660', 'technician...","{'name': 'ABDUL TOLIB', 'id': 7099, 'technicia...",PS_PM_QTR_StationInspection_40000530204.pdf,12660,09/05/2023,7099,09/05/2023
2,4000627745,2024-09-18 00:00:00,"{'start_time': '00:54:00', 'end_time': '09:23:...","{'name': 'SUFFI SHUKHAIRI', 'id': 20971, 'tech...","{'name': 'SUFFI SHUKHAIRI', 'id': 7094, 'techn...",PS_PM_WEK_StationInspection_4000627745.pdf,20971,18/09/2024,7094,18/09/2024
3,4000627751,2024-09-20 00:00:00,"{'start_time': '00:00:00', 'end_time': '01:30:...","{'name': 'IZZUDDIN', 'id': '22388', 'technicia...","{'name': 'NORIEZUWADI', 'id': '19234', 'techni...",PS_PM_WEK_StationInspection_4000627751.pdf,22388,20/09/2024,19234,20/09/2024
4,4000462020,2022-04-25 00:00:00,"{'start_time': '1000', 'end_time': '1100', 'to...","{'name': 'ALI AKHBAR', 'id': '7013', 'technici...","{'name': 'ABDUL TOLIB', 'id': '7099', 'technic...",PS_PM_WEK_StationInspection_4000462020.pdf,7013,25/04/2022,7099,25/04/2022


In [ ]:
id_to_name = dict(zip(
    cleaned_user["stamp_id"].astype(str),
    cleaned_user["name"]
))

id_to_supervisor_name = id_to_name

def build_rows(row):
    
    tech_ids = row["technician_ids"]

    if not isinstance(tech_ids, str):
        return []

    tech_list = [i.strip() for i in tech_ids.split(",") if i.strip()]

    results = []

    for tech_id in tech_list:
        
        results.append({
            "filename": row.get("filename"),
            "workorder_no": row.get("workorder_no"),
            "plan_start_date_time": row.get("plan_start_date_time"),
            "plan_end_date_time": row.get("plan_end_date_time"),

            "technician_id": tech_id,
            "name": id_to_name.get(tech_id, ""),

            "supervisor_id": row.get("supervisor_id"),
            "supervisor_name": id_to_supervisor_name.get(
                str(row.get("supervisor_id")), ""
            ),
        })

    return results

expanded = df_psd.apply(build_rows, axis=1).explode().dropna()

final_df = pd.DataFrame(expanded.tolist())
final_df.head()

In [ ]:
final_df.to_excel("output/staff_psd.xlsx", index=False)